In [ ]:
# %% [markdown]
# # 🧪 Notebook 01: Data Exploration & Featurization
# **Objective:** Inspect raw PDB files and test the Graph featurization logic.

# %%
import sys
import os
import torch
import networkx as nx
import matplotlib.pyplot as plt
from pathlib import Path

# Add src to path
sys.path.append(str(Path(os.getcwd()).parent))

from src.ingestion.parser import ProteinParser
from src.chemistry.feature_extraction import MoleculeFeaturizer

# %% [markdown]
# ## 1. Inspect Protein Data
# Let's load a PDB file and visualize the extracted backbone.

# %%
# Ensure you have run 'src/pipeline/run_ingestion.py' first!
pdb_path = "../data/raw/proteins/5R82.pdb" 

if os.path.exists(pdb_path):
    parser = ProteinParser()
    coords = parser.parse_pdb(pdb_path)
    
    print(f"Protein: 5R82")
    print(f"Extracted {coords.shape[0]} alpha-carbon residues.")
    
    # 3D Scatter Plot of the backbone
    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c='blue', alpha=0.6)
    ax.set_title("Protein Backbone (Alpha Carbons)")
    plt.show()
else:
    print("⚠️ Data not found. Please run the ingestion pipeline.")

# %% [markdown]
# ## 2. Inspect Chemical Graph
# Convert a SMILES string into the graph format our GNN expects.

# %%
smiles = "CC(=O)Oc1ccccc1C(=O)O" # Aspirin
featurizer = MoleculeFeaturizer()
x, edge_index = featurizer.process_smiles(smiles)

print(f"Drug: Aspirin")
print(f"Node Features: {x.shape}")
print(f"Edges: {edge_index.shape}")

# Convert to NetworkX for 2D visualization
g = nx.Graph()
edges = edge_index.t().tolist()
g.add_edges_from(edges)

plt.figure(figsize=(6, 6))
pos = nx.spring_layout(g)
nx.draw(g, pos, with_labels=True, node_color='lightgreen', node_size=500)
plt.title("Molecular Graph Representation")
plt.show()